# Бэйзлайн (дефолтные параметры) на реальных датасетах

Алгоритмы запускаются с параметрами по умолчанию - без подбора сетки.

In [1]:
import sys, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    adjusted_rand_score, adjusted_mutual_info_score,
    normalized_mutual_info_score, fowlkes_mallows_score,
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
)

sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
from data_generator.uci_real import (
    load_iris_fc, load_wine_fc, load_ecoli_fc, load_seeds_fc, load_segment_fc,
)


In [2]:
from algorithms import (
    DBSCANWrapper, HDBSCANWrapper,
    DPCWrapper, RDDACWrapper, CKDPCWrapper,
)

ALG_NAMES = ['DBSCAN', 'HDBSCAN', 'DPC', 'RD-DAC', 'CKDPC']
ALG_CLASSES = {
    'DBSCAN':  DBSCANWrapper,
    'HDBSCAN': HDBSCANWrapper,
    'DPC':     DPCWrapper,
    'RD-DAC':  RDDACWrapper,
    'CKDPC':   CKDPCWrapper,
}


In [3]:
DATASETS = [load_iris_fc(), load_wine_fc(), load_ecoli_fc(), load_seeds_fc(), load_segment_fc()]

In [4]:
def compute_all_metrics(X, labels, y_true):
    labels  = np.asarray(labels, dtype=int)
    y_true  = np.asarray(y_true, dtype=int)
    mask_nn = labels != -1
    k_found   = len(set(labels[mask_nn].tolist())) if mask_nn.any() else 0
    noise_pct = float((~mask_nn).sum()) / len(labels) * 100
    if k_found >= 2:
        ari = float(adjusted_rand_score(y_true, labels))
        ami = float(adjusted_mutual_info_score(y_true, labels))
        nmi = float(normalized_mutual_info_score(y_true, labels))
        fmi = float(fowlkes_mallows_score(y_true, labels))
    elif k_found == 1:
        ari = ami = nmi = fmi = 0.0
    else:
        ari = ami = nmi = fmi = float('nan')
    X_sub, l_sub = X[mask_nn], labels[mask_nn]
    if mask_nn.sum() >= 2 and len(np.unique(l_sub)) >= 2:
        try:    sc  = float(silhouette_score(X_sub, l_sub))
        except: sc  = float('nan')
        try:    chi = float(calinski_harabasz_score(X_sub, l_sub))
        except: chi = float('nan')
        try:    dbi = float(davies_bouldin_score(X_sub, l_sub))
        except: dbi = float('nan')
    else:
        sc = chi = dbi = float('nan')
    return dict(k_found=k_found, noise_pct=noise_pct,
                ARI=ari, AMI=ami, NMI=nmi, FMI=fmi, SC=sc, CHI=chi, DBI=dbi)


def run_default(alg_class, X):
    try:
        return np.asarray(alg_class().fit_predict(X), dtype=int)
    except Exception as e:
        print(f'  ERROR: {e}')
        return np.zeros(len(X), dtype=int)


In [36]:
def show_results(all_results, datasets):
    summary = []
    for ds in datasets:
        dname = ds.name
        k_true = len(np.unique(ds.y_true))
        for alg in ALG_NAMES:
            m = all_results[dname][alg]
            def _v(k): return round(m[k], 4) if isinstance(m[k], float) and m[k]==m[k] else None
            summary.append({'Dataset': dname, 'k_true': k_true,
                            'Algorithm': alg, 'k_found': m['k_found'],
                            'noise%': round(m['noise_pct'],1),
                            'ARI': _v('ARI'), 'AMI': _v('AMI'), 'NMI': _v('NMI'),
                            'FMI': _v('FMI'), 'SC': _v('SC'), 'CHI': _v('CHI'), 'DBI': _v('DBI')})
    df = pd.DataFrame(summary)
    display(df.set_index(['Dataset','Algorithm']).style
            .format({'ARI':'{:.4f}','AMI':'{:.4f}','NMI':'{:.4f}',
                     'FMI':'{:.4f}','SC':'{:.4f}','DBI':'{:.4f}',
                     'noise%':'{:.1f}'}, na_rep='-')
            .background_gradient(subset=['ARI', 'AMI', 'NMI', 'FMI'], cmap='RdYlGn', vmin=0, vmax=1)
            .set_caption('Полная таблица метрик'))

In [37]:
ALL_RESULTS = {}
for ds in DATASETS:
    X, y = ds.X, ds.y_true
    print(f'\n{ds.name}  [{X.shape[0]} x {X.shape[1]}]  k_true={len(np.unique(y))}')
    rows = {}
    for alg_name in ALG_NAMES:
        t0     = time.time()
        labels = run_default(ALG_CLASSES[alg_name], X)
        mets   = compute_all_metrics(X, labels, y)
        elapsed = time.time() - t0
        rows[alg_name] = mets
        print(f'  {alg_name:<8}  k={mets["k_found"]}  noise={mets["noise_pct"]:.1f}%  '
              f'ARI={mets["ARI"]:.4f}  ({elapsed:.1f}s)')
    ALL_RESULTS[ds.name] = rows



uci_iris  [150 x 4]  k_true=3
  DBSCAN    k=6  noise=41.3%  ARI=0.4151  (0.0s)
  HDBSCAN   k=2  noise=0.0%  ARI=0.5681  (0.0s)
  DPC       k=3  noise=0.0%  ARI=0.7196  (0.0s)
  RD-DAC    k=3  noise=0.0%  ARI=0.6844  (0.0s)
  CKDPC     k=3  noise=6.7%  ARI=0.5310  (0.0s)

uci_wine  [178 x 13]  k_true=3
  DBSCAN    k=4  noise=42.1%  ARI=0.2462  (0.0s)
  HDBSCAN   k=2  noise=15.2%  ARI=0.3917  (0.0s)
  DPC       k=3  noise=0.0%  ARI=0.6724  (0.0s)
  RD-DAC    k=4  noise=0.0%  ARI=0.6387  (0.0s)
  CKDPC     k=2  noise=9.0%  ARI=-0.0064  (0.0s)

uci_ecoli  [327 x 7]  k_true=5
  DBSCAN    k=5  noise=35.2%  ARI=0.3441  (0.0s)
  HDBSCAN   k=5  noise=37.9%  ARI=0.3243  (0.0s)
  DPC       k=3  noise=0.0%  ARI=0.6654  (0.0s)
  RD-DAC    k=5  noise=0.0%  ARI=0.7203  (0.0s)
  CKDPC     k=3  noise=3.7%  ARI=0.0208  (0.0s)

uci_seeds  [210 x 7]  k_true=3
  DBSCAN    k=5  noise=48.6%  ARI=0.2469  (0.0s)
  HDBSCAN   k=4  noise=19.5%  ARI=0.3328  (0.0s)
  DPC       k=4  noise=0.0%  ARI=0.6354  (0.0s)
 

In [38]:
show_results(ALL_RESULTS, DATASETS)